In [1]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

# --------- Paths (edit this root path once) ----------
PROJECT_ROOT = Path(r"C:\DataAnalysis\repos\project-01-transport-expenses")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --------- SQL connection ----------
server = "localhost,1433"
database = "TransportAnalysis"
username = "sa"
password = "StrongPass!123"
driver = "ODBC Driver 18 for SQL Server"

engine = create_engine(
    f"mssql+pyodbc://{username}:{password}@{server}/{database}"
    f"?driver={driver}&TrustServerCertificate=yes"
)

# --------- 1) Extract ----------
df = pd.read_sql("SELECT * FROM dbo.presto_fake_data;", engine)

# --------- 2) Clean / enforce types ----------
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# if Time comes in as object/string, force it
df["Time"] = pd.to_datetime(df["Time"].astype(str), format="%H:%M:%S", errors="coerce").dt.time

df["Hour"] = pd.to_numeric(df["Hour"], errors="coerce").astype("Int64")
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

# --------- 3) Feature engineering ----------
df["IsPaidFare"] = (df["Transaction_Type"] == "Fare Payment").astype(int)
df["MonthStart"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["WeekdayNum"] = df["Date"].dt.weekday  # 0=Mon

df["DateTime"] = pd.to_datetime(df["Date"].dt.date.astype(str) + " " + df["Hour"].astype(str).str.zfill(2) + ":00:00")

# --------- 4) Basic validation (cheap & important) ----------
bad_dates = df["Date"].isna().sum()
bad_time = df["Time"].isna().sum()
if bad_dates or bad_time:
    print(f"⚠️ Bad dates: {bad_dates}, bad times: {bad_time}")

# --------- 5) Save processed outputs ----------
# Best format for analytics
processed_parquet = PROCESSED_DIR / "presto_processed.parquet"
df.to_parquet(processed_parquet, index=False)

# Optional: CSV for easy viewing/Excel/Tableau
processed_csv = PROCESSED_DIR / "presto_processed.csv"
df.to_csv(processed_csv, index=False)

print("Saved:", processed_parquet)
print("Saved:", processed_csv)


Saved: C:\DataAnalysis\repos\project-01-transport-expenses\data\processed\presto_processed.parquet
Saved: C:\DataAnalysis\repos\project-01-transport-expenses\data\processed\presto_processed.csv


In [2]:
paid = df[df["Transaction_Type"] == "Fare Payment"].copy()

monthly_spend = (paid.groupby("MonthStart", as_index=False)["Amount"].sum()
                 .rename(columns={"Amount": "MonthlySpend"}))

peak_hours = (paid.groupby("Hour", as_index=False)
              .size()
              .rename(columns={"size": "PaidTrips"}))

top_locations = (paid.groupby("Location", as_index=False)
                 .size()
                 .rename(columns={"size": "PaidTrips"})
                 .sort_values("PaidTrips", ascending=False)
                 .head(15))

(monthly_spend.to_csv(PROCESSED_DIR / "kpi_monthly_spend.csv", index=False))
(peak_hours.to_csv(PROCESSED_DIR / "kpi_peak_hours.csv", index=False))
(top_locations.to_csv(PROCESSED_DIR / "kpi_top_locations.csv", index=False))


In [3]:
df.head()

,Date,Year,Month,Weekday,Time,Hour,Transaction_Type,Amount,Location,Commute_Tag,IsPaidFare,MonthStart,WeekdayNum,DateTime
0,2025-12-23,2025,December,Tuesday,15:10:00,15,Fare Payment,4.12,LESLIE ST / HIGHWAY 7,Work Commute,1,2025-12-01,1,2025-12-23 15:00:00
1,2025-12-01,2025,December,Monday,07:15:00,7,Fare Payment,4.12,HIGH TECH RD / RED MAPLE RD,Errand,1,2025-12-01,0,2025-12-01 07:00:00
2,2025-12-06,2025,December,Saturday,06:15:00,6,Free Transfer,0.00,YONGE ST / 16TH AVE,Work Commute,0,2025-12-01,5,2025-12-06 06:00:00
3,2025-10-16,2025,October,Thursday,07:30:00,7,Fare Payment,4.12,RICHMOND HILL CENTRE,Errand,1,2025-10-01,3,2025-10-16 07:00:00
4,2025-11-22,2025,November,Saturday,08:40:00,8,Fare Payment,4.12,DON MILLS STATION,Errand,1,2025-11-01,5,2025-11-22 08:00:00


In [4]:
df.describe()

,Date,Year,Hour,Amount,IsPaidFare,MonthStart,WeekdayNum,DateTime
count,219,219.0,219.0,219.000000,219.000000,219,219.000000,219
mean,2025-11-14 08:32:52.602739712,2025.0,12.13242,2.803105,0.680365,2025-10-30 06:21:22.191780864,3.365297,2025-11-14 20:40:49.315068416
min,2025-10-02 00:00:00,2025.0,6.0,0.000000,0.000000,2025-10-01 00:00:00,0.000000,2025-10-02 15:00:00
25%,2025-10-19 00:00:00,2025.0,7.0,0.000000,0.000000,2025-10-01 00:00:00,2.000000,2025-10-19 15:00:00
50%,2025-11-16 00:00:00,2025.0,15.0,4.120000,1.000000,2025-11-01 00:00:00,3.000000,2025-11-16 17:00:00
75%,2025-12-07 00:00:00,2025.0,16.0,4.120000,1.000000,2025-12-01 00:00:00,5.000000,2025-12-07 16:00:00
max,2025-12-30 00:00:00,2025.0,18.0,4.120000,1.000000,2025-12-01 00:00:00,6.000000,2025-12-30 15:00:00
std,NaN,0.0,4.744461,1.925702,0.467403,NaN,2.070740,NaN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              219 non-null    datetime64[ns]
 1   Year              219 non-null    int64         
 2   Month             219 non-null    object        
 3   Weekday           219 non-null    object        
 4   Time              219 non-null    object        
 5   Hour              219 non-null    Int64         
 6   Transaction_Type  219 non-null    object        
 7   Amount            219 non-null    float64       
 8   Location          219 non-null    object        
 9   Commute_Tag       219 non-null    object        
 10  IsPaidFare        219 non-null    int64         
 11  MonthStart        219 non-null    datetime64[ns]
 12  WeekdayNum        219 non-null    int32         
 13  DateTime          219 non-null    datetime64[ns]
dtypes: Int64(1), datetime64[ns